# 🏎️ F1 Driver–Circuit Compatibility
## Notebook 01 — Data Collection

**⚠️ NOTE: You can skip this notebook entirely.**  
The output CSV files (`laps_raw.csv`, `results_raw.csv`, `schedule_raw.csv`) are already included in `data/raw/`.  
Go directly to **Notebook 02** unless you want to re-collect the data from scratch.

---

**If you want to re-collect from scratch:**
1. Delete the CSV files in `data/raw/` and `data/processed/`
2. Run this notebook — it will re-fetch everything from the FastF1 API
3. Then run NB02 onwards as normal

---

**Crash-safe — saves after EVERY race:**
- ✅ Saves after every single race — if it crashes, rerun and it picks up exactly where it left off
- ✅ 15 second sleep between races to avoid FastF1 rate limits (500 calls/hour)
- ✅ 120 second pause + 5 retries when rate limit is hit

**Output:** `data/raw/laps_raw.csv`, `data/raw/results_raw.csv`, `data/raw/schedule_raw.csv`

> ⏱️ First run: ~90–120 min. Subsequent runs (cache exists): ~2 min.

In [ ]:
%pip install fastf1 --quiet

In [ ]:
import fastf1
import pandas as pd
import numpy as np
import os
import time
import warnings
warnings.filterwarnings('ignore')

print(f"✅ FastF1 version: {fastf1.__version__}")

## Step 1 — Configure Paths

In [ ]:
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))

PATHS = {
    'raw':       os.path.join(PROJECT_ROOT, 'data', 'raw'),
    'processed': os.path.join(PROJECT_ROOT, 'data', 'processed'),
    'cache':     os.path.join(PROJECT_ROOT, 'data', 'cache'),
    'figures':   os.path.join(PROJECT_ROOT, 'figures'),
    'models':    os.path.join(PROJECT_ROOT, 'models'),
}

for name, path in PATHS.items():
    os.makedirs(path, exist_ok=True)
    print(f"✅ {name}: {path}")

fastf1.Cache.enable_cache(PATHS['cache'])
print(f"\n✅ Cache enabled at: {PATHS['cache']}")

## Step 2 — Configuration

> 💡 15 seconds between races is safe. Increase to 20+ if you keep hitting rate limits.

In [ ]:
SEASONS               = [2019, 2020, 2021, 2022, 2023, 2024, 2025]
SLEEP_BETWEEN_RACES   = 15   # seconds between each race — safe for FastF1 rate limits
SLEEP_ON_RATE_LIMIT   = 120  # seconds to pause when rate limit detected
MAX_RETRIES           = 5    # retries per race before giving up

LAP_COLS = [
    'Season', 'RoundNumber', 'CircuitName', 'EventName',
    'Driver', 'Team',
    'LapNumber', 'LapTime',
    'Sector1Time', 'Sector2Time', 'Sector3Time',
    'Compound', 'TyreLife', 'Stint',
    'PitInTime', 'PitOutTime',
    'TrackStatus', 'IsPersonalBest'
]

RESULT_COLS = [
    'Season', 'RoundNumber', 'CircuitName', 'EventName',
    'Abbreviation', 'FullName', 'TeamName',
    'GridPosition', 'Position',
    'Points', 'Status'
]

print("✅ Config ready.")
print(f"   Seasons: {SEASONS}")
print(f"   Sleep between races: {SLEEP_BETWEEN_RACES}s")
print(f"   Rate limit pause: {SLEEP_ON_RATE_LIMIT}s")
print(f"   Max retries per race: {MAX_RETRIES}")

## Step 3 — Collect Schedule

In [ ]:
all_schedules = []

for season in SEASONS:
    try:
        schedule = fastf1.get_event_schedule(season, include_testing=False)
        schedule['Season'] = season
        all_schedules.append(schedule)
        print(f"✅ {season}: {len(schedule)} events")
        time.sleep(2)
    except Exception as e:
        print(f"❌ {season}: {e}")

schedule_df = pd.concat(all_schedules, ignore_index=True)
schedule_df.to_csv(os.path.join(PATHS['raw'], 'schedule_raw.csv'), index=False)
print(f"\n💾 schedule_raw.csv saved — {len(schedule_df)} rows")

## Step 4 — Helper Functions

In [ ]:
def is_rate_limit_error(error_msg):
    keywords = ['500 calls', 'rate limit', 'too many requests', '429', 'ratelimit']
    return any(k.lower() in str(error_msg).lower() for k in keywords)


def load_session_with_retry(season, round_number, max_retries=MAX_RETRIES):
    """Load a race session with retry + rate-limit backoff."""
    for attempt in range(1, max_retries + 1):
        try:
            session = fastf1.get_session(season, round_number, 'R')
            session.load(telemetry=False, weather=False, messages=False)

            laps = session.laps.copy()
            laps['Season']      = season
            laps['RoundNumber'] = round_number
            laps['CircuitName'] = session.event['Location']
            laps['EventName']   = session.event['EventName']
            laps = laps[[c for c in LAP_COLS if c in laps.columns]]

            results = session.results.copy()
            results['Season']      = season
            results['RoundNumber'] = round_number
            results['CircuitName'] = session.event['Location']
            results['EventName']   = session.event['EventName']
            results = results[[c for c in RESULT_COLS if c in results.columns]]

            return laps, results

        except Exception as e:
            if is_rate_limit_error(str(e)):
                print(f"     ⚠️  Rate limit! Pausing {SLEEP_ON_RATE_LIMIT}s... (attempt {attempt}/{max_retries})")
                time.sleep(SLEEP_ON_RATE_LIMIT)
            else:
                print(f"     ⚠️  Attempt {attempt}/{max_retries} failed: {str(e)[:80]}")
                if attempt < max_retries:
                    time.sleep(15)

    return None, None


def load_existing_csv(path):
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"   📂 Loaded existing: {os.path.basename(path)} ({len(df):,} rows)")
        return df
    return pd.DataFrame()


print("✅ Helper functions ready.")

## Step 5 — Main Collection Loop

> **Crash-safe:** saves after every single race. If it crashes or hits a rate limit, just re-run this cell — it automatically skips every race already collected and picks up where it left off.

In [ ]:
laps_path    = os.path.join(PATHS['raw'], 'laps_raw.csv')
results_path = os.path.join(PATHS['raw'], 'results_raw.csv')

print("🔍 Checking for existing data...")
laps_master    = load_existing_csv(laps_path)
results_master = load_existing_csv(results_path)

# Track completed races as (season, round) pairs — not just seasons
if not laps_master.empty and 'Season' in laps_master.columns and 'RoundNumber' in laps_master.columns:
    completed_races = set(
        zip(laps_master['Season'].tolist(), laps_master['RoundNumber'].tolist())
    )
    print(f"   ✅ Already collected: {len(completed_races)} races")
else:
    completed_races = set()
    print("   📭 Starting fresh.")

all_failed = []

for season in SEASONS:
    print(f"\n{'='*55}")
    print(f"📅 SEASON {season}")
    print(f"{'='*55}")

    try:
        schedule = fastf1.get_event_schedule(season, include_testing=False)
        time.sleep(2)
    except Exception as e:
        print(f"  ❌ Could not fetch schedule: {e}")
        continue

    total_races  = len(schedule)
    season_failed = []

    for idx, (_, event) in enumerate(schedule.iterrows(), start=1):
        round_num    = event['RoundNumber']
        circuit_name = event.get('Location', 'Unknown')

        # Skip already collected races
        if (season, round_num) in completed_races:
            print(f"  [{idx:02d}/{total_races:02d}] {circuit_name} ... ⏭️  skipping (already collected)")
            continue

        print(f"  [{idx:02d}/{total_races:02d}] {circuit_name} ... ", end="", flush=True)

        laps, results = load_session_with_retry(season, round_num)

        if laps is not None:
            # Save after every single race
            laps_master    = pd.concat([laps_master,    laps],    ignore_index=True)
            results_master = pd.concat([results_master, results], ignore_index=True)
            laps_master.to_csv(laps_path,    index=False)
            results_master.to_csv(results_path, index=False)
            completed_races.add((season, round_num))
            print(f"✅ {len(laps)} laps | total so far: {len(laps_master):,} rows")
        else:
            print(f"❌ Failed after {MAX_RETRIES} retries")
            season_failed.append({'season': season, 'round': round_num, 'circuit': circuit_name})

        time.sleep(SLEEP_BETWEEN_RACES)

    all_failed.extend(season_failed)
    print(f"\n  ✅ Season {season} done — {len(season_failed)} races failed")

print(f"\n{'='*55}")
print("🏁 Collection complete!")
print(f"   Total laps rows:    {len(laps_master):,}")
print(f"   Total results rows: {len(results_master):,}")
print(f"   Total failed races: {len(all_failed)}")

## Step 6 — Review Failures

If any races failed, just re-run Step 5 — it skips everything already collected and only retries the failed ones.

In [ ]:
if all_failed:
    print(f"⚠️  {len(all_failed)} races failed:")
    for f in all_failed:
        print(f"   {f['season']} Round {f['round']:02d} — {f['circuit']}")
    print("\n💡 Re-run Step 5 — completed races will be skipped automatically.")
else:
    print("✅ No failed races!")

## Step 7 — Sanity Check

In [ ]:
laps_check    = pd.read_csv(laps_path)
results_check = pd.read_csv(results_path)

print("📊 Final file sizes:")
print(f"   laps_raw:    {laps_check.shape[0]:,} rows × {laps_check.shape[1]} cols")
print(f"   results_raw: {results_check.shape[0]:,} rows × {results_check.shape[1]} cols")

print("\n📅 Races collected per season:")
print(results_check.groupby('Season')['RoundNumber'].nunique().rename('races').to_string())

print("\n🏎️  Drivers per season:")
print(results_check.groupby('Season')['Abbreviation'].nunique().rename('drivers').to_string())

In [ ]:
laps_check.head(5)